# 02 Image Captioning

- Implement image captioning
- Uses BLIP to generate captions from input images

## Setup

In [2]:
# Install required libraries
!pip install pandas transformers gTTS torch torchvision pillow

  Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Generate captions

In [4]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from transformers import BlipForConditionalGeneration, BlipProcessor

# Paths
images_dir = Path("../coco_indoor")
output_dir = Path("captions")
output_dir.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load processor and model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# Load image files
image_files = sorted([f for f in images_dir.iterdir() if f.suffix.lower() in ['.jpg']])

# Generate caption for an image
def generate_caption(image_path, max_new_tokens=20):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    caption = processor.decode(outputs[0], skip_special_tokens=True)
    return caption

results = []
for image_file in image_files:
    try:
        caption = generate_caption(image_file)
        results.append({
            "image": image_file.name,
            "caption": caption
        })
        print(f"{image_file.name}: {caption}")
    except Exception as e:
        results.append({
            "image": image_file.name,
            "caption": None,
            "error": str(e)
        })

captions_df = pd.DataFrame(results)
captions_df.to_csv(output_dir / "blip_captions.csv", index=False)

captions_df.head()

Loading weights: 100%|██████████| 473/473 [00:00<00:00, 33892.06it/s]


000000000139.jpg: a living room with a television and a fireplace
000000000632.jpg: a bedroom with a bed and a window
000000000802.jpg: a kitchen with a white refrigerator and a white stove
000000001353.jpg: a group of people riding on a red train
000000001993.jpg: a bedroom with a bed and a table
000000002685.jpg: a group of people standing around a kitchen
000000003156.jpg: a man sitting on a toilet
000000003934.jpg: a woman and a child playing with a remote
000000004495.jpg: a living room with a couch and a television
000000005193.jpg: a group of people standing around a surf board
000000006213.jpg: a bathroom with a sink, mirror, and shower
000000006818.jpg: a bathroom with a toilet and a bucket
000000007574.jpg: a kitchen with a sink and a refrigerator
000000007795.jpg: two beds in a room
000000009483.jpg: a man standing in front of a desk with a laptop
000000009590.jpg: a group of people sitting around a table
000000010363.jpg: a cat standing on top of a car
000000012576.jpg: a p

,image,caption
0,000000000139.jpg,a living room with a television and a fireplace
1,000000000632.jpg,a bedroom with a bed and a window
2,000000000802.jpg,a kitchen with a white refrigerator and a whit...
3,000000001353.jpg,a group of people riding on a red train
4,000000001993.jpg,a bedroom with a bed and a table
